# Does the reply give a counterexample?

This pilot uses ten matched training pairs from distinct OP discussions. Each opening is scored individually with its original post. The seeded sample and all review inputs exclude delta outcomes.

A counterexample is a specific actual or hypothetical case that conflicts with the OP's main claim or shows that the claim needs to be narrowed. Use `present` when you can quote both the claim and the specific case, `absent` for clear cases without a counterexample, and `unclear` when the claim-to-example relationship is ambiguous. Judge how the reply argues, taking its example as stated.

Fill in the human columns in the blind review CSV before looking at the model's answers. The notebook allows you to read one pair at a time.

In [1]:
from html import escape
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

root = next(parent for parent in (Path.cwd(), *Path.cwd().parents)
            if (parent / "project/notebooks/linguistic-features/argument-mining").is_dir())
result_root = root / "project/data/processed/linguistic-features/counterexample-pilot"
run_id = None  # Set a run folder name to open a particular sample.
runs = sorted(path.parent for path in result_root.glob("*/pilot_manifest.json"))
if run_id is None and not runs:
    raise FileNotFoundError("Run run_counterexample_pilot.py --prepare-only to create a blinded pilot.")
run = result_root / run_id if run_id else runs[-1]
manifest = json.loads((run / "pilot_manifest.json").read_text())
review_path = run / "blind_counterexample_review.csv"
review = pd.read_csv(review_path, keep_default_na=False)
print(f'{manifest["pairs"]} training pairs · {manifest["replies"]} openings · seed {manifest["seed"]}')
print(review_path)

10 training pairs · 20 openings · seed 42
/Users/paolacalle/Desktop/NYU/semesters/fall-2026/ml-viz/ml-visualization-project/ml-visualization-project/project/data/processed/linguistic-features/counterexample-pilot/20261121T225229997731Z/blind_counterexample_review.csv


In [2]:
pair = "C01"  # Choose C01 through C10.
replies = review.loc[review["review_id"].str.startswith(f"{pair}-")]
if len(replies) != 2:
    raise ValueError("Choose a pilot pair from C01 through C10.")

def show_text(label, text):
    display(HTML(f'<h4>{escape(label)}</h4><div style="white-space: pre-wrap">{escape(text)}</div>'))

first = replies.iloc[0]
show_text(first["op_title"], first["op_body"])
for reply in replies.itertuples(index=False):
    show_text(reply.review_id, reply.opening_reply)

## Model and rules

Qwen3-8B runs locally from the official 4-bit GGUF checkpoint. It returns a label and two quotes for each `present` decision. The span check tests whether the output contains actual source text after normalizing whitespace. Human review determines whether the passages support the decision.

The rule baseline flags cue phrases such as “for instance” or “what about”. We retain the phrases for inspection. The rule does not check how the passage relates to the OP claim.

In [3]:
prediction_path = run / "model_counterexample_predictions.csv"
if not prediction_path.exists():
    print("Model predictions will appear after local scoring completes.")
else:
    predictions = pd.read_csv(prediction_path, keep_default_na=False)
    display(predictions.groupby(["label", "span_check"]).size().rename("replies").to_frame())
    valid_present = predictions["label"].eq("present") & predictions["span_check"].eq("spans_found")
    valid_absent = predictions["label"].eq("absent") & predictions["span_check"].eq("no_spans_expected")
    pair_counts = predictions.assign(present=valid_present, usable=valid_present | valid_absent).groupby(
        predictions["review_id"].str.split("-").str[0]
    )[["present", "usable"]].sum()
    pattern = pair_counts["present"].map({0: "Neither reply", 1: "Exactly one reply", 2: "Both replies"})
    pattern = pattern.where(pair_counts["usable"].eq(2), "Unresolved pair")
    display(pattern.value_counts().rename("pairs").to_frame())
    review = pd.read_csv(review_path, keep_default_na=False)
    review["human_label"] = review["human_label"].str.strip().str.lower()
    if not review["human_label"].isin(["", "present", "absent", "unclear"]).all():
        raise ValueError("Enter present, absent, or unclear in human_label.")
    labeled = review.loc[review["human_label"].ne("")]
    if labeled.empty:
        print("Add human labels to compare the detectors with your review.")
    else:
        check = labeled.merge(predictions, on="review_id", validate="one_to_one")
        display(pd.crosstab(check["human_label"], check["label"].replace("", "invalid"), dropna=False))
        resolved = check.loc[check["human_label"].isin(["present", "absent"])]
        model_resolved = resolved.loc[
            resolved["label"].isin(["present", "absent"])
            & resolved["span_check"].isin(["spans_found", "no_spans_expected"])
        ]
        print(f"Model gave a usable decision on {len(model_resolved)} of {len(resolved)} human-resolved replies.")
        measurements = []
        for name, subset, predicted in [
            ("Cue rule", model_resolved, model_resolved["rule_cues"].ne("")),
            ("Qwen3-8B", model_resolved, model_resolved["label"].eq("present")),
        ]:
            truth = subset["human_label"].eq("present")
            tp = int((truth & predicted).sum())
            fp = int((~truth & predicted).sum())
            fn = int((truth & ~predicted).sum())
            measurements.append({
                "detector": name, "resolved replies assessed": len(subset),
                "TP": tp, "FP": fp, "FN": fn,
                "precision": tp / (tp + fp) if tp + fp else None,
                "recall": tp / (tp + fn) if tp + fn else None,
            })
        display(pd.DataFrame(measurements).set_index("detector").round(3))
        display(check[["review_id", "human_label", "label", "op_claim", "reply_span", "span_check", "rule_cues"]])

replies
label   span_check                        
present span_missing_or_unmatched        3
        spans_found                     17

,pairs
present,
Both replies,8
Unresolved pair,2


Add human labels to compare the detectors with your review.
